# Bike-Demand - Colab runner (train -> per-city eval)

HUJI IML 67577, Challenge 1. Clone -> install -> load data -> local time-split -> train ->
per-city MAE -> save outputs to Drive. Runs top-to-bottom on a fresh Colab VM.

**Data source (Section 4):**
- `"enriched"` (default) - the **full-year, multi-city enriched** data, auto-downloaded from the
  GitHub Release (no Drive needed). The full year is ~16M rides; **free Colab (~12GB, no High-RAM)
  can't fit that**, so Section 4 has a **`MONTHS`** knob - keep the first N months (default 4) to fit.
  Use the **ensemble** + **`FAST=True`**. Set `MONTHS=12` only on a High-RAM runtime.
- `"official"` - the provided `train_set.csv` from your Google Drive (cities 1-3).

**Notes**
- **GPU optional:** the XGBoost ensemble **auto-detects a GPU** (`Runtime > Change runtime type > GPU`)
  and uses it to speed up training; CPU works fine too. (The LightGBM baseline is CPU-only.)
  Note: building the big station-hour table is CPU/RAM-bound, so **High-RAM helps more than GPU**.
- The VM is **ephemeral**, so outputs save to **Drive**.
- For `enriched`: `Runtime > Change runtime type > High-RAM`. For a quick test, edit `FILES`
  in Section 4 to one city (e.g. just `la_2025_enriched.csv.gz`).

In [ ]:
# === Section 2: clone the repo ===========================================
REPO_URL = "https://github.com/GuyShabat7/HUJI-IML-Hackathon.git"  # <- or your fork
BRANCH   = "main"

import os, subprocess
REPO_DIR = "/content/HUJI-IML-Hackathon"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
subprocess.run(["git", "log", "--oneline", "-1"])

In [ ]:
# === Section 3: install dependencies =====================================
!pip -q install lightgbm xgboost joblib pandas numpy scikit-learn
import lightgbm, xgboost, joblib, pandas, numpy, sklearn
print("lightgbm", lightgbm.__version__, "| xgboost", xgboost.__version__,
      "| pandas", pandas.__version__, "| numpy", numpy.__version__, "| sklearn", sklearn.__version__)

In [ ]:
# === Section 4: load training data =======================================
DATA_SOURCE = "enriched"          # "enriched" (full-year, from Release) | "official" (Drive)
REPO        = "GuyShabat7/HUJI-IML-Hackathon"
FILES       = ["london_2025_enriched.csv.gz", "dc_2025_enriched.csv.gz", "la_2025_enriched.csv.gz"]
MONTHS      = 4   # keep months 1..MONTHS (demand stays correct). FREE Colab ~12GB RAM: use 3-4.
                  # MONTHS=12 = full year, but the 16M-row table build needs a High-RAM runtime.

import os, glob, shutil, subprocess
os.makedirs("dataset", exist_ok=True)
DST = "dataset/train_set.csv"

if DATA_SOURCE == "enriched":
    import pandas as pd
    base = f"https://github.com/{REPO}/releases/download/enriched-2025-data"
    keep = set(range(1, MONTHS + 1))
    for fn in FILES:
        if not (os.path.exists(fn) and os.path.getsize(fn) > 1000):
            print("downloading", fn, "...")
            subprocess.run(["wget", "-q", f"{base}/{fn}"], check=True)
    # chunked, month-filtered concat -> low memory (never loads a whole city at once)
    n, first = 0, True
    with open(DST, "w", newline="") as out:
        for fn in FILES:
            for ch in pd.read_csv(fn, compression="gzip", chunksize=500_000, dtype=str):
                ch = ch[ch["started_at"].str[5:7].astype(int).isin(keep)]
                ch.to_csv(out, index=False, header=first); first = False; n += len(ch)
    print(f"enriched rows (months 1-{MONTHS}): {n:,} -> {DST}")
else:
    DRIVE_CSV = ""                        # "" = auto-find train_set.csv in your Drive
    from google.colab import drive
    drive.mount("/content/drive")
    src = DRIVE_CSV if (DRIVE_CSV and os.path.exists(DRIVE_CSV)) else None
    if not src:
        hits = glob.glob("/content/drive/MyDrive/**/train_set.csv", recursive=True)
        src = hits[0] if hits else None
    if src:
        shutil.copy(src, DST); print("using", src)
    else:
        print("not in Drive - upload it:")
        from google.colab import files
        up = files.upload(); open(DST, "wb").write(next(iter(up.values())))

import pandas as pd
print("cities:", sorted(pd.read_csv(DST, usecols=["city"]).city.unique()))

In [ ]:
# === Section 5: local time-split + build evaluator-format files ==========
# Time-based holdout per city (latest ~20%), then the station-hour eval format.
!python make_local_split.py
!python build_station_hour_eval_data.py --input_csv dataset/local_validation_set.csv --public_targets_csv dataset/public_validation_targets.csv --private_labels_csv dataset/private_labels.csv

import os
for f in ["dataset/local_train_set.csv", "dataset/local_validation_set.csv",
          "dataset/public_validation_targets.csv", "dataset/private_labels.csv"]:
    print("ok " if os.path.exists(f) else "MISSING ", f)

In [ ]:
# === Section 6: train ====================================================
SUBMISSION = "submissions/challenge_1_ensamble"  # XGBoost ensemble (recommended for enriched)
FAST       = True                                 # small trees; honored by the ensemble. Set False for full recipe.
#   official-data baseline instead: SUBMISSION="submissions/challenge_1_IDs", FAST=False

import os, time, subprocess
cmd = ["python", "train.py"]
if FAST:
    cmd.append("--fast")   # the ensemble honors --fast; the LightGBM baseline simply ignores it
print("running:", " ".join(cmd), "in", SUBMISSION)
t0 = time.time()
subprocess.run(cmd, cwd=SUBMISSION, check=True)
print()
print(f"trained {SUBMISSION} in {time.time()-t0:.1f}s")
assert os.path.exists(os.path.join(SUBMISSION, "weights.joblib")), "train.py did not produce weights.joblib"

In [ ]:
# === Section 7: per-city MAE =============================================
# Official evaluator on the local validation targets built in Section 5.
# (It scans submissions/; only folders with a weights.joblib score - others are skipped.)
!python evaluate.py --eval_dir dataset --submissions_dir submissions --test_targets_csv public_validation_targets.csv --test_labels_csv private_labels.csv --output_csv mae_by_city.csv

import pandas as pd
print()
print(pd.read_csv("mae_by_city.csv").to_string(index=False))

# NOTE: train.py trains on the FULL train_set.csv, so this validation slice overlaps training
# (optimistic). For an honest held-out per-city + blended number (model trained on c1+c2,
# city 3 unseen), use the repo harness instead:
# !python tools/eval_local.py {SUBMISSION}

In [ ]:
# === Section 8: save outputs to Drive ====================================
import os, shutil
OUT_DIR = "/content/drive/MyDrive/iml_hackathon/outputs"
os.makedirs(OUT_DIR, exist_ok=True)
saved = []
w = os.path.join(SUBMISSION, "weights.joblib")
if os.path.exists(w):
    dst = os.path.join(OUT_DIR, f"weights_{os.path.basename(SUBMISSION)}.joblib")
    shutil.copy(w, dst); saved.append(dst)
if os.path.exists("mae_by_city.csv"):
    dst = os.path.join(OUT_DIR, "mae_by_city.csv")
    shutil.copy("mae_by_city.csv", dst); saved.append(dst)
print("saved to Drive:")
for s in saved:
    print("  ", s)